# SQL Basics Practice Exercises

**Estimated time:** ~6 hours (part of the ~10 hour sql-basics level, alongside `sql-basics-guide.ipynb`).

These exercises follow `sql-basics-guide.ipynb` in order, and every heading names the **guide section** it
practises — so when an exercise stumps you, re-read that section first. (Numbering starts at 3 because guide
sections 1 and 2 are the setup and the schema tour, which the setup cell below covers.)

## How to Use This Notebook

- **Run the setup cell first.** It builds the shop database in memory from `../assets/sql/` and defines `q()` and `run()`.
- Read the instructions in each markdown cell, write your SQL between the triple quotes in the code cell below
  it, and run it with **Shift + Enter**.
- Every cell ends with `assert` checks. Your solution is correct when the cell runs and prints `OK` with no
  `AssertionError`.
- **Use the exact column names and aliases the instructions ask for** — the checks compare them.
- The instructions always give you an `ORDER BY` to use. That is not decoration: without it the row order is
  undefined and the checks cannot be trusted.
- Try each exercise before looking anything up. When you are stuck, run your query without the checks and look
  at what comes back — the shape of a wrong answer usually tells you which clause is wrong.
- If a check fails, read the assert to see what was expected, then print your result next to it.
- Nothing you do here can damage anything. Re-run the setup cell and the database is new again.

## Setup — Run This First

This builds the shop database in memory from the bundled CSV files and gives you the two helpers used
throughout: `q(sql)`
returns a query result as a DataFrame, `run(sql)` executes statements that change data. Run it once. If you
restart the kernel, run it again.

In [3]:
import sqlite3
import pandas as pd

SQL = "assets/sql"          # the bundled schema and CSV files
TABLES = ["categories", "customers", "employees", "products",
          "orders", "order_items", "payments"]

con = sqlite3.connect(":memory:")         # the database lives in RAM -- nothing to clean up

with open(f"{SQL}/schema.sql") as f:
    con.executescript(f.read())           # creates the seven empty tables

con.execute("PRAGMA foreign_keys = ON")   # from here on, SQLite enforces the foreign keys

for table in TABLES:
    pd.read_csv(f"{SQL}/{table}.csv").to_sql(table, con, if_exists="append", index=False)
con.commit()


def q(sql):
    """Run a SELECT and hand the result back as a pandas DataFrame."""
    return pd.read_sql_query(sql, con)


def run(sql):
    """Run statements that change data or structure: CREATE, INSERT, UPDATE, DELETE."""
    con.executescript(sql)
    con.commit()


pd.set_option("display.width", 110)
pd.set_option("display.max_rows", 25)

for table in TABLES:
    print(f"{table:12s} {q(f'SELECT COUNT(*) AS n FROM {table}')['n'][0]:>4} rows")

categories      8 rows
customers      60 rows
employees      15 rows
products       40 rows
orders        300 rows
order_items   673 rows
payments      248 rows


## Exercise 3: SELECT and Aliases  *(guide section 3)*

Return the `product_id`, the product name aliased as **`product`**, and the `price` for every product in
category 3 (Audio), ordered by `price` descending and then `product_id`.

Your query should produce exactly the columns `product_id`, `product`, `price`.

In [6]:
sql = """
Select product_id , name as product , price
from products
where category_id==3
order by price desc , product_id
"""

out = q(sql)

assert list(out.columns) == ["product_id", "product", "price"]
assert len(out) == 6
assert out.round(2).values.tolist() == [
    [14, "Halo Studio Headset", 18900],
    [13, "Halo Over-Ear", 11500],
    [16, "Quiet Desk Mic", 7600],
    [12, "Echo Buds Pro", 6800],
    [15, "Rumble Bluetooth Speaker", 4500],
    [11, "Echo Buds", 3200],
]
print("OK")

OK


## Exercise 4: Filtering with WHERE  *(guide section 4)*

Two separate questions, each in its own query.

1. `cheap` — the `name` and `price` of every product priced **under 2000**, ordered by `price` then `name`.
2. `chennai` — the `customer_id` and `name` of every customer whose `city` is exactly `'Chennai'`, ordered by
   `customer_id`.

Text values go in single quotes, and the comparison is case-sensitive.

In [11]:
cheap_sql = """
SELECT name , price 
FROM products
WHERE price < 2000
ORDER BY price , name
"""
cheap = q(cheap_sql)

chennai_sql = """
SELECT  customer_id , name 
FROM customers
WHERE  city='Chennai'
ORDER BY customer_id
"""
chennai = q(chennai_sql)

assert list(cheap.columns) == ["name", "price"]
assert len(cheap) == 7
assert cheap.round(2).values.tolist()[0] == ["Braid USB-C Cable 1m", 450]
assert cheap.round(2).values.tolist()[-1] == ["Carry 256GB Flash Drive", 1950]
assert round(float(cheap["price"].sum()), 2) == 8450
assert list(chennai.columns) == ["customer_id", "name"]
assert len(chennai) == 12
assert chennai.round(2).values.tolist()[0] == [8, "Kiran Nair"]
assert chennai.round(2).values.tolist()[-1] == [48, "Zoya Das"]
assert round(float(chennai["customer_id"].sum()), 2) == 335
print("OK")

OK


## Exercise 5: Combining Conditions  *(guide section 5)*

Return the `name`, `category_id`, `price` and `stock` of every product that is

- in category **4** (Accessories) **or** category **6** (Storage), **and**
- priced **at or above 1000**, **and**
- has **more than 100** in stock.

Order by `price` descending, then `name`. Watch the bracket around the `OR`.

In [15]:
sql = """
SELECT p.name, p.category_id, p.price, p.stock
FROM products AS p
JOIN categories AS c
    ON p.category_id = c.category_id
WHERE (c.name = 'Accessories' OR c.name = 'Storage')
  AND p.price >= 1000
  AND p.stock > 100
ORDER BY p.price DESC, p.name; 

"""

out = q(sql)

assert list(out.columns) == ["name", "category_id", "price", "stock"]
assert len(out) == 5
assert out.round(2).values.tolist() == [
    ["Anchor 100W Charger", 4, 2900, 130],
    ["Glide Pro Mouse", 4, 2400, 150],
    ["Carry 256GB Flash Drive", 6, 1950, 180],
    ["Anchor 65W Charger", 4, 1800, 210],
    ["Carry 128GB Flash Drive", 6, 1150, 240],
]
print("OK")

OK


## Exercise 6: BETWEEN, IN and LIKE  *(guide section 6)*

Three shorthand filters.

1. `mid` — `name` and `price` of products priced **between 5000 and 12000 inclusive**, using `BETWEEN`.
   Order by `price`, then `name`.
2. `picked` — `name` and `category_id` of products in categories **1, 5 or 7**, using `IN`. Order by
   `category_id`, then `name`.
3. `phones` — `name` of every product whose name **contains** the word `Phone`, using `LIKE`. Order by `name`.

In [17]:
mid_sql = """
SELECT name , price 
FROM products
WHERE price BETWEEN 5000 and 12000
ORDER BY price , name
"""
mid = q(mid_sql)

picked_sql = """
SELECT name ,category_id 
FROM products
WHERE category_id IN (1,5,7)
ORDER BY category_id , name
"""
picked = q(picked_sql)

phones_sql = """
SELECT name 
FROM products
WHERE name LIKE '%Phone%'
ORDER BY name
"""
phones = q(phones_sql)

assert list(mid.columns) == ["name", "price"]
assert len(mid) == 8
assert mid.round(2).values.tolist()[0] == ["Vault 1TB SSD", 6400]
assert mid.round(2).values.tolist()[-1] == ["Clarity 24 Monitor", 11800]
assert round(float(mid["price"].sum()), 2) == 72500
assert list(picked.columns) == ["name", "category_id"]
assert len(picked) == 12
assert picked.round(2).values.tolist()[0] == ["Aster 14 Laptop", 1]
assert picked.round(2).values.tolist()[-1] == ["Frame Mirrorless Body", 7]
assert round(float(picked["category_id"].sum()), 2) == 46
assert list(phones.columns) == ["name"]
assert len(phones) == 5
assert phones.round(2).values.tolist() == [
    ["Orbit One Phone"],
    ["Orbit Pro Phone"],
    ["Orbit Ultra Phone"],
    ["Pixi Lite Phone"],
    ["Pixi Plus Phone"],
]
print("OK")

OK


## Exercise 7: NULL  *(guide section 7)*

The customers table has five rows with no `city`.

1. `missing` — `customer_id` and `name` of every customer whose `city` is unknown, ordered by `customer_id`.
   Use the right operator; `= NULL` will return nothing.
2. `counts` — a **single row** with three columns: `everyone` (all customers), `not_mumbai` (customers whose
   `city <> 'Mumbai'`) and `not_mumbai_or_unknown` (customers whose city is not Mumbai **or** is unknown).
   Use three scalar subqueries, or three `SUM(CASE ...)` expressions over one pass — either is fine.

The gap between the last two columns is the point of the exercise.

In [18]:
missing_sql = """
SELECT customer_id , name 
FROM customers
WHERE city IS NULL 
ORDER BY customer_id
"""
missing = q(missing_sql)

counts_sql = """
SELECT (SELECT COUNT(*) FROM customers) as everyone,
(SELECT COUNT(*) FROM customers WHERE city <> 'Mumbai') as not_mumbai,
(SELECT COUNT(*) FROM customers WHERE city <> 'Mumbai' OR city IS NULL) as not_mumbai_or_unknown
"""
counts = q(counts_sql)

assert list(missing.columns) == ["customer_id", "name"]
assert len(missing) == 5
assert missing.round(2).values.tolist() == [
    [7, "Kabir Bose"],
    [19, "Varun Pillai"],
    [33, "Eshan Gupta"],
    [44, "Ekta Menon"],
    [52, "Harish Reddy"],
]
assert list(counts.columns) == ["everyone", "not_mumbai", "not_mumbai_or_unknown"]
assert len(counts) == 1
assert counts.round(2).values.tolist() == [[60, 46, 51]]
print("OK")

OK


## Exercise 8: Sorting  *(guide section 8)*

Return the `name`, `category_id` and `stock` of the **12 products with the most stock**, ordered by `stock`
descending with `name` as the tiebreaker.

The tiebreaker is doing real work here — several products share a stock level, so without it the twelfth row is
not reproducible.

In [21]:
sql = """
SELECT name , category_id , stock
FROM products
ORDER BY stock DESC , name
LIMIT 12
"""

out = q(sql)

assert list(out.columns) == ["name", "category_id", "stock"]
assert len(out) == 12
assert out.round(2).values.tolist() == [
    ["Braid USB-C Cable 1m", 4, 500],
    ["Braid USB-C Cable 2m", 4, 420],
    ["Glide Wireless Mouse", 4, 300],
    ["Carry 128GB Flash Drive", 6, 240],
    ["Anchor 65W Charger", 4, 210],
    ["Carry 256GB Flash Drive", 6, 180],
    ["Tick Fitness Band", 8, 165],
    ["Glide Pro Mouse", 4, 150],
    ["Echo Buds", 3, 140],
    ["Anchor 100W Charger", 4, 130],
    ["Rumble Bluetooth Speaker", 3, 110],
    ["Clack Mini Keyboard", 4, 95],
]
print("OK")

OK


## Exercise 9: Paging with LIMIT and OFFSET  *(guide section 9)*

Sort all products by `price` descending with `name` as the tiebreaker, then hand back two pages of six.

1. `page1` — `name` and `price`, rows 1 to 6.
2. `page3` — `name` and `price`, rows 13 to 18.

Both queries need the same `ORDER BY`, or the pages will not line up.

In [22]:
page1_sql = """
SELECT name , price
FROM products
ORDER BY price DESC , name
LIMIT 6
"""
page1 = q(page1_sql)

page3_sql = """
SELECT name , price
FROM products
ORDER BY price DESC , name
LIMIT 6 OFFSET 12
"""
page3 = q(page3_sql)

assert list(page1.columns) == ["name", "price"]
assert len(page1) == 6
assert page1.round(2).values.tolist() == [
    ["Vega Book 16 Studio", 142000],
    ["Nimbus Air Laptop", 118000],
    ["Aster 15 Pro Laptop", 94000],
    ["Frame Mirrorless Body", 86000],
    ["Orbit Ultra Phone", 79000],
    ["Aster 14 Laptop", 62000],
]
assert list(page3.columns) == ["name", "price"]
assert len(page3) == 6
assert page3.round(2).values.tolist() == [
    ["Tick Watch Ultra", 26500],
    ["Clarity 27 QHD Monitor", 21500],
    ["Pixi Plus Phone", 19900],
    ["Halo Studio Headset", 18900],
    ["Frame 50mm Lens", 15400],
    ["Pixi Lite Phone", 13500],
]
print("OK")

OK


## Exercise 10: DISTINCT  *(guide section 10)*

1. `methods` — every distinct payment `method`, ordered by `method`.
2. `tallies` — a **single row** with three columns: `order_rows` (the number of rows in `orders`),
   `distinct_customers` (how many different customers appear in `orders`) and `distinct_days` (how many
   different `order_date` values appear).

For the second one, remember that `COUNT(DISTINCT column)` is a single expression.

In [ ]:
methods_sql = """
-- Your SQL here
"""
methods = q(methods_sql)

tallies_sql = """
-- Your SQL here
"""
tallies = q(tallies_sql)

assert list(methods.columns) == ["method"]
assert len(methods) == 5
assert methods.round(2).values.tolist() == [["card"], ["cod"], ["netbanking"], ["upi"], ["wallet"]]
assert list(tallies.columns) == ["order_rows", "distinct_customers", "distinct_days"]
assert len(tallies) == 1
assert tallies.round(2).values.tolist() == [[300, 51, 250]]
print("OK")

## Exercise 11: Computed Columns  *(guide section 11)*

For every product in category 1 (Laptops) return

- `name`
- `price`
- `margin` — `price - cost`
- `margin_pct` — the margin as a percentage of `price`, rounded to **1** decimal place

Order by `margin_pct` descending, then `name`.

If every `margin_pct` comes back as a whole number, you have hit integer division — look again at what you
multiplied by.

In [29]:
sql = """
SELECT  name , price , (price-cost) AS margin , ROUND((price-cost)*100.0/price,1) AS margin_pct
FROM products
WHERE category_id is 1
ORDER BY margin_pct DESC , name
LIMIT 5
"""

out = q(sql)

assert list(out.columns) == ["name", "price", "margin", "margin_pct"]
assert len(out) == 5
assert out.round(2).values.tolist() == [
    ["Vega Book 13", 54000, 10000, 18.5],
    ["Aster 14 Laptop", 62000, 11000, 17.7],
    ["Aster 15 Pro Laptop", 94000, 16000, 17],
    ["Nimbus Air Laptop", 118000, 19000, 16.1],
    ["Vega Book 16 Studio", 142000, 21000, 14.8],
]
print("OK")

OK


## Exercise 12: Text Functions  *(guide section 12)*

For the first eight customers by `customer_id` return

- `customer_id`
- `initial` — the **first character** of `name`
- `name_length` — how many characters are in `name`
- `domain` — everything in `email` **after** the `@`
- `label` — the name, then a space, then the city in brackets, like `Nisha Joshi (Mumbai)`

Order by `customer_id` and limit to 8. Use `SUBSTR`, `LENGTH`, `INSTR` and `||`.

Watch what happens to `label` for a customer with no city — that is `NULL` behaving exactly as section 7
described, and the check expects it.

In [33]:
sql = """
SELECT customer_id , SUBSTR(name,1,1) as initial , LENGTH(name) as name_length , 
 SUBSTR(email,INSTR(email,'@')+1) as domain , name || ' (' || city || ')' AS label
 FROM customers
 ORDER BY customer_id 
 LIMIT 8
"""

out = q(sql)

assert list(out.columns) == ["customer_id", "initial", "name_length", "domain", "label"]
assert len(out) == 8
assert out.round(2).fillna("<NULL>").values.tolist() == [
    [1, "N", 11, "mail.com", "Nisha Joshi (Mumbai)"],
    [2, "M", 11, "example.com", "Manoj Menon (Mumbai)"],
    [3, "M", 12, "example.com", "Mohit Pillai (Kochi)"],
    [4, "P", 11, "example.com", "Pooja Singh (Delhi)"],
    [5, "V", 12, "inbox.in", "Varun Chopra (Kochi)"],
    [6, "H", 9, "mail.com", "Hema Khan (Mumbai)"],
    [7, "K", 10, "inbox.in", "<NULL>"],
    [8, "K", 10, "mail.com", "Kiran Nair (Chennai)"],
]
print("OK")

OK


## Exercise 13: Dates  *(guide section 13)*

1. `q4` — `order_id`, `order_date` and `status` for every order placed in **October, November or December
   2024**, ordered by `order_date` then `order_id`. Use a date range, not `strftime`.
2. `by_year` — one row per calendar year with columns `year` and `orders`, ordered by `year`. Use
   `strftime('%Y', order_date)`.

In [34]:
q4_sql = """
SELECT order_id, order_date, status
FROM orders
WHERE order_date >= '2024-10-01'
  AND order_date < '2025-01-01'
ORDER BY order_date, order_id
"""
q4 = q(q4_sql)

by_year_sql = """
SELECT
    strftime('%Y', order_date) AS year,
    COUNT(*) AS orders
FROM orders
GROUP BY strftime('%Y', order_date)
ORDER BY year
"""
by_year = q(by_year_sql)

assert list(q4.columns) == ["order_id", "order_date", "status"]
assert len(q4) == 66
assert q4.round(2).values.tolist()[0] == [241, "2024-10-02", "delivered"]
assert q4.round(2).values.tolist()[-1] == [285, "2024-12-31", "shipped"]
assert round(float(q4["order_id"].sum()), 2) == 17655
assert list(by_year.columns) == ["year", "orders"]
assert len(by_year) == 2
assert by_year.round(2).values.tolist() == [["2023", 115], ["2024", 185]]
print("OK")

OK


## Exercise 14: Aggregates  *(guide section 14)*

Return a **single row** summarising the `order_items` table, with these columns:

- `line_items` — how many rows there are
- `total_units` — the sum of `quantity`
- `avg_units` — the average `quantity`, rounded to 2 decimals
- `biggest_line` — the largest `quantity`
- `revenue` — the sum of `quantity * unit_price * (1 - discount)`, rounded to 2 decimals

That last expression is how revenue is calculated everywhere in this course, so it is worth getting into your
fingers.

In [36]:
sql = """
SELECT count(*) as line_items , SUM(quantity) as total_units , ROUND(AVG(quantity),2) as avg_units , MAX(quantity) as biggest_line,
SUM(ROUND(quantity*unit_price*(1-discount),2)) as revenue
from order_items
"""

out = q(sql)

assert list(out.columns) == ["line_items", "total_units", "avg_units", "biggest_line", "revenue"]
assert len(out) == 1
assert out.round(2).values.tolist() == [[673, 1009, 1.5, 5, 23782962.5]]
print("OK")

OK


## Exercise 15: GROUP BY  *(guide section 15)*

One row per order `status`, with

- `status`
- `orders` — how many orders have it
- `customers` — how many **distinct** customers have an order with it
- `first_seen` — the earliest `order_date` in that status
- `last_seen` — the latest

Order by `orders` descending, then `status`.

In [39]:
sql = """
SELECT  status , count(*) as orders , count(DISTINCT customer_id) as customers , MIN(order_date) as first_seen , 
MAX(order_date) as last_seen
FROM orders
GROUP BY status
ORDER BY orders DESC , status
"""

out = q(sql)

assert list(out.columns) == ["status", "orders", "customers", "first_seen", "last_seen"]
assert len(out) == 5
assert out.round(2).values.tolist() == [
    ["delivered", 176, 46, "2023-01-10", "2024-12-31"],
    ["shipped", 50, 31, "2023-01-10", "2024-12-31"],
    ["placed", 39, 26, "2023-01-26", "2024-12-31"],
    ["cancelled", 18, 15, "2023-02-10", "2024-12-27"],
    ["returned", 17, 13, "2023-03-29", "2024-12-03"],
]
print("OK")

OK


## Exercise 16: WHERE vs HAVING  *(guide section 16)*

Using `order_items` only: one row per `product_id`, but only for products that have been **ordered on more
than 20 separate lines**, and counting only lines with a `quantity` of at least 2.

Columns: `product_id`, `lines` (the number of qualifying rows), `units` (the sum of `quantity`).
Order by `lines` descending, then `product_id`.

Decide carefully which of the two conditions belongs in `WHERE` and which in `HAVING`.

In [45]:
sql = """
SELECT product_id , COUNT(*) AS lines , SUM(quantity) as units 
FROM order_items
WHERE quantity>=2
GROUP BY product_id
HAVING COUNT(*)>20 
ORDER BY lines DESC, product_id

"""

out = q(sql)

assert list(out.columns) == ["product_id", "lines", "units"]
assert len(out) == 0
assert out.round(2).values.tolist() == []
print("OK")

OK


## Exercise 17: CASE WHEN  *(guide section 17)*

One row per `channel`, with

- `channel`
- `orders` — total orders on that channel
- `delivered` — how many of them have status `'delivered'`
- `cancelled` — how many have status `'cancelled'`
- `delivered_pct` — `delivered` as a percentage of `orders`, rounded to 1 decimal

Order by `delivered_pct` descending, then `channel`.

Use `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` for the two conditional counts, so the whole thing is one pass over
the table.

In [47]:
sql = """
SELECT channel , count(*) as orders , SUM(CASE 
WHEN status='delivered' THEN 1 ELSE 0 END) as delivered,
 SUM(CASE 
WHEN status='cancelled' THEN 1 ELSE 0 END) as cancelled,
ROUND(100*AVG(CASE WHEN status='delivered' THEN 1 ELSE 0 END),1) as delivered_pct
FROM orders
GROUP BY channel
ORDER BY delivered_pct DESC , channel 
"""

out = q(sql)

assert list(out.columns) == ["channel", "orders", "delivered", "cancelled", "delivered_pct"]
assert len(out) == 4
assert out.round(2).values.tolist() == [
    ["web", 114, 76, 5, 66.7],
    ["store", 62, 36, 4, 58.1],
    ["phone", 31, 17, 2, 54.8],
    ["app", 93, 47, 7, 50.5],
]
print("OK")

OK


## Exercise 18: COALESCE and NULLIF  *(guide section 18)*

Return one row per customer for the **ten lowest `customer_id`s**, with

- `customer_id`
- `city_label` — the city, or the text `'unknown'` when it is missing
- `state_label` — the state, or the text `'unknown'` when it is missing
- `has_city` — `1` when the city is filled in, `0` when it is not

Order by `customer_id` and limit to 10. Use `COALESCE` for the labels and a `CASE` for the flag.

In [50]:
sql = """
SELECT customer_id , COALESCE(city,'unknown') as city_label , COALESCE(state,'unknown') as state_label,
 (CASE WHEN city is null THEN 0 ELSE 1 END) as has_city
 FROM customers
 ORDER BY customer_id
 limit 10
 """

out = q(sql)

assert list(out.columns) == ["customer_id", "city_label", "state_label", "has_city"]
assert len(out) == 10
assert out.round(2).values.tolist() == [
    [1, "Mumbai", "Maharashtra", 1],
    [2, "Mumbai", "Maharashtra", 1],
    [3, "Kochi", "Kerala", 1],
    [4, "Delhi", "Delhi", 1],
    [5, "Kochi", "Kerala", 1],
    [6, "Mumbai", "Maharashtra", 1],
    [7, "unknown", "unknown", 0],
    [8, "Chennai", "Tamil Nadu", 1],
    [9, "Pune", "Maharashtra", 1],
    [10, "Mumbai", "Maharashtra", 1],
]
print("OK")

OK


## Exercise 19: INNER JOIN  *(guide section 19)*

Join `products` to `categories` and return, for each **category**, the number of products and the average
price:

- `category` — the category's `name`
- `products` — how many products are in it
- `avg_price` — their average `price`, rounded to the nearest whole number

Order by `avg_price` descending, then `category`.

Both tables have a column called `name`, so alias the tables and prefix every column.

In [51]:
sql = """
select c.name as category , count(*) as products , ROUND(AVG(p.price)) as avg_price
FROM products as p join categories as c on p.category_id=c.category_id 
GROUP by c.name
ORDER BY avg_price DESC , category

"""

out = q(sql)

assert list(out.columns) == ["category", "products", "avg_price"]
assert len(out) == 8
assert out.round(2).values.tolist() == [
    ["Laptops", 5, 94000],
    ["Cameras", 3, 45133],
    ["Phones", 5, 37480],
    ["Monitors", 4, 31825],
    ["Wearables", 3, 13067],
    ["Audio", 6, 8750],
    ["Storage", 5, 5600],
    ["Accessories", 9, 2128],
]
print("OK")

OK


## Exercise 20: LEFT JOIN  *(guide section 20)*

1. `never_ordered` — `product_id` and `name` of every product that has **never** appeared in `order_items`.
   Use a `LEFT JOIN` and test for the missing side. Order by `product_id`.
2. `unpaid_by_status` — one row per order `status` counting the orders that have **no** matching row in
   `payments`. Columns `status` and `unpaid`, ordered by `unpaid` descending then `status`.

Both are the same shape: `LEFT JOIN`, then `WHERE <right table primary key> IS NULL`.

In [54]:
never_ordered_sql = """
SELECT
    p.product_id,
    p.name
FROM products AS p
LEFT JOIN order_items AS oi
    ON p.product_id = oi.product_id
WHERE oi.product_id IS NULL
ORDER BY p.product_id;
"""
never_ordered = q(never_ordered_sql)

unpaid_by_status_sql = """
SELECT
    o.status,
    COUNT(*) AS unpaid
FROM orders AS o
LEFT JOIN payments AS p
    ON o.order_id = p.order_id
WHERE p.order_id IS NULL
GROUP BY o.status
ORDER BY unpaid DESC, o.status; 
"""
unpaid_by_status = q(unpaid_by_status_sql)

assert list(never_ordered.columns) == ["product_id", "name"]
assert len(never_ordered) == 2
assert never_ordered.round(2).values.tolist() == [[25, "Stand Up Laptop Riser"], [33, "Carry 128GB Flash Drive"]]
assert list(unpaid_by_status.columns) == ["status", "unpaid"]
assert len(unpaid_by_status) == 3
assert unpaid_by_status.round(2).values.tolist() == [["cancelled", 18], ["placed", 17], ["returned", 17]]
print("OK")

OK


## Exercise 21: Every Clause at Once  *(guide section 21)*

One query using `FROM`, `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY` and `LIMIT` together.

From `orders`, ignoring anything with status `'cancelled'`, return the **three busiest months of 2024** that
had **more than 12 orders**:

- `month` — `strftime('%Y-%m', order_date)`
- `orders` — how many orders that month

Order by `orders` descending, then `month`. Limit to 3.

In [55]:
sql = """
select strftime('%Y-%m', order_date) as month , count(*) as orders
from orders 
where order_date >= '2024-01-01'
  and order_date < '2025-01-01'
  and status <> 'cancelled'
group by strftime('%Y-%m', order_date)
having count(*)>12
order by orders desc , month
limit 3

"""

out = q(sql)

assert list(out.columns) == ["month", "orders"]
assert len(out) == 3
assert out.round(2).values.tolist() == [["2024-11", 23], ["2024-12", 23], ["2024-07", 17]]
print("OK")

OK


## Exercise 22: Creating and Changing Data  *(guide section 22)*

Build a small table of your own, using `run()` for the statements that change things and `q()` to check.

1. Create a table called `supplier` with these columns:
   - `supplier_id INTEGER PRIMARY KEY`
   - `name TEXT NOT NULL`
   - `city TEXT`
   - `rating INTEGER NOT NULL DEFAULT 3` with a `CHECK` that keeps it between 1 and 5
2. Insert these four rows **in this order**, leaving `rating` to its default on the last one:
   `('Northwind', 'Delhi', 4)`, `('Southgate', 'Chennai', 2)`, `('Eastline', 'Kolkata', 5)`,
   `('Westford', 'Mumbai', <let the default fill it in>)`
3. `UPDATE` Westford's rating to 4.
4. `DELETE` every supplier whose rating is below 4 — that should remove exactly one row, Southgate.

Start with `DROP TABLE IF EXISTS supplier;` so the cell can be run twice.

In [62]:
run("""DROP TABLE IF EXISTS supplier;

CREATE TABLE supplier(
supplier_id INTEGER PRIMARY KEY,
name TEXT NOT NULL,
city TEXT,
rating INTEGER NOT NULL DEFAULT 3 CHECK(rating between 1 and 5)
);
INSERT INTO supplier(supplier_id,name,city,rating)
VALUES (1,'Northwind', 'Delhi', 4), (2,'Southgate', 'Chennai', 2),(3,'Eastline', 'Kolkata', 5);
INSERT INTO supplier(supplier_id, name, city)
VALUES
    (4, 'Westford', 'Mumbai');
UPDATE supplier
SET rating=4
WHERE name='Westford';

DELETE FROM supplier
WHERE rating<4;

""")

left = q("SELECT * FROM supplier ORDER BY supplier_id")

assert list(left.columns) == ["supplier_id", "name", "city", "rating"]
assert len(left) == 3, "one supplier should have been deleted"
assert left["name"].tolist() == ["Northwind", "Eastline", "Westford"]
assert left["rating"].tolist() == [4, 5, 4], "Westford took the DEFAULT of 3, then your UPDATE made it 4"
assert q("SELECT COUNT(*) AS n FROM supplier WHERE rating < 4")["n"][0] == 0
print("OK")

OK


## Exercise 23: SQL or pandas  *(guide section 23)*

Answer the same question twice and prove the answers match.

**The question:** how many orders did each `status` get in 2024?

1. `in_sql` — do the filtering, grouping and sorting **in SQL**. Columns `status` and `orders`, ordered by
   `orders` descending then `status`, with a clean `0..n` index.
2. `in_pandas` — pull `SELECT status, order_date FROM orders` into a DataFrame and do the same work with pandas
   (`[]`, `groupby`, `sort_values`). End with the same two columns, same order, and
   `.reset_index(drop=True)` so the index matches.

The final check compares the two frames directly.

In [63]:
in_sql = q("""
select status,
    COUNT(*) AS orders
FROM orders
WHERE order_date >= '2024-01-01'
  AND order_date < '2025-01-01'
GROUP BY status
ORDER BY orders DESC, status
""")

orders_df = q("SELECT status, order_date FROM orders")
# Your pandas here
in_pandas = (    orders_df[
        (orders_df["order_date"] >= "2024-01-01") &
        (orders_df["order_date"] < "2025-01-01")
    ].groupby("status").size().reset_index(name="orders").sort_values(["orders", "status"], ascending=[False, True]).reset_index(drop=True))

assert list(in_sql.columns) == ["status", "orders"]
assert len(in_sql) == 5
assert in_sql.round(2).values.tolist() == [["delivered", 102], ["shipped", 35], ["placed", 22], ["cancelled", 13], ["returned", 13]]
assert in_sql.equals(in_pandas), "same question, so the two frames should be identical"
print("OK")

OK


## Exercise 24: Spotting the Pitfalls  *(guide section 25)*

One row that demonstrates four of the traps from section 25 side by side. Query `customers` and return:

- `everyone` — the total number of customers
- `bengaluru` — how many have `city = 'Bengaluru'`
- `not_bengaluru` — how many satisfy `city <> 'Bengaluru'`
- `unknown_city` — how many have no city
- `adds_up` — `1` if `bengaluru + not_bengaluru + unknown_city` equals `everyone`, else `0`

Use scalar subqueries or conditional sums, whichever you prefer. The point is to see the numbers next to each
other and know why `bengaluru + not_bengaluru` does not reach `everyone`.

In [69]:
sql = """
select count(*) as everyone, sum(case when city='Bengaluru' then 1 else 0 end) as bengaluru,
sum(case when city<>'Bengaluru' then 1 else 0 end) as not_bengaluru ,
sum(case when city is null then 1 else 0 end) as unknown_city,
 case
        WHEN
            SUM(CASE WHEN city = 'Bengaluru' THEN 1 ELSE 0 END)
          + SUM(CASE WHEN city <> 'Bengaluru' THEN 1 ELSE 0 END)
          + SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END)
          = COUNT(*)
        THEN 1
        ELSE 0
    END AS adds_up
from customers

"""

out = q(sql)

assert list(out.columns) == ["everyone", "bengaluru", "not_bengaluru", "unknown_city", "adds_up"]
assert len(out) == 1
assert out.round(2).values.tolist() == [[60, 10, 45, 5, 1]]
print("OK")

OK


## Exercise 25: Mini Project — A Monthly Sales Report  *(mini project)*

Build the report a manager would actually ask for: **revenue by month for 2024**, excluding cancelled orders.

Join `orders` to `order_items` and return one row per month with

- `month` — `strftime('%Y-%m', o.order_date)`
- `orders` — the number of **distinct** orders that month (not the number of item lines)
- `units` — total `quantity`
- `revenue` — `SUM(quantity * unit_price * (1 - discount))`, rounded to 2 decimals
- `avg_order_value` — `revenue / orders`, rounded to 2 decimals

Order by `month`.

Two things decide whether this is right. First, once you join to `order_items` an order appears once per line,
so counting orders needs `COUNT(DISTINCT o.order_id)`. Second, `avg_order_value` must divide by that distinct
count, not by the row count.

In [73]:
sql = """
SELECT
    strftime('%Y-%m', o.order_date) AS month,
    COUNT(DISTINCT o.order_id) AS orders,
    SUM(oi.quantity) AS units,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)),2 ) AS revenue,
    ROUND(
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount))
    / COUNT(DISTINCT o.order_id),
    2
) AS avg_order_value
FROM orders AS o
JOIN order_items AS oi
ON o.order_id = oi.order_id
WHERE o.order_date >= '2024-01-01'
  AND o.order_date < '2025-01-01'
  AND o.status <> 'cancelled'
GROUP BY strftime('%Y-%m', o.order_date)
ORDER BY month
"""

out = q(sql)

assert list(out.columns) == ["month", "orders", "units", "revenue", "avg_order_value"]
assert len(out) == 12
assert out.round(2).values.tolist() == [
    ["2024-01", 8, 26, 414130, 51766.25],
    ["2024-02", 7, 21, 315950, 45135.71],
    ["2024-03", 15, 50, 983710, 65580.67],
    ["2024-04", 6, 19, 321250, 53541.67],
    ["2024-05", 16, 53, 1246130, 77883.13],
    ["2024-06", 11, 32, 920957.5, 83723.41],
    ["2024-07", 17, 46, 796725, 46866.18],
    ["2024-08", 17, 80, 1690255, 99426.76],
    ["2024-09", 14, 56, 2164965, 154640.36],
    ["2024-10", 15, 57, 1250365, 83357.67],
    ["2024-11", 23, 81, 2328947.5, 101258.59],
    ["2024-12", 23, 73, 1945537.5, 84588.59],
]
print("OK")

OK


## Exercise 26: Mini Project — A Customer Directory  *(mini project)*

Build a directory row for every customer, whether or not they have ever ordered. One row per customer, with

- `customer_id`
- `name`
- `city_label` — the city, or `'unknown'`
- `orders` — how many orders they have placed (0 for the nine who never have)
- `last_order` — their most recent `order_date`, or `'never'` for those nine
- `segment` — `'repeat'` when they have 3 or more orders, `'once'` when they have 1 or 2, `'prospect'` when
  they have none

Order by `orders` descending, then `customer_id`. Limit to the first 15 rows.

This pulls together nearly everything in the level: a `LEFT JOIN` so nobody is lost, `COUNT` of the right
column so the prospects score zero, `COALESCE` for two different kinds of missing, a `CASE` over an aggregate,
and a deterministic sort.

In [75]:
sql = """
SELECT
    c.customer_id,
    c.name,
    COALESCE(c.city, 'unknown') AS city_label,
    COUNT(o.order_id) AS orders,
    COALESCE(MAX(o.order_date), 'never') AS last_order,
    CASE
WHEN COUNT(o.order_id) >= 3 THEN 'repeat'
WHEN COUNT(o.order_id) >= 1 THEN 'once'
ELSE 'prospect'
END AS segment
FROM customers AS c
LEFT JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.city
ORDER BY orders DESC, c.customer_id
LIMIT 15
"""

out = q(sql)

assert list(out.columns) == ["customer_id", "name", "city_label", "orders", "last_order", "segment"]
assert len(out) == 15
assert out.round(2).values.tolist() == [
    [6, "Hema Khan", "Mumbai", 21, "2024-11-20", "repeat"],
    [59, "Neha Reddy", "Mumbai", 19, "2024-12-28", "repeat"],
    [9, "Bhavya Menon", "Pune", 14, "2024-12-07", "repeat"],
    [27, "Bhavya Das", "Chennai", 12, "2024-12-31", "repeat"],
    [34, "Zara Mehta", "Hyderabad", 12, "2024-12-09", "repeat"],
    [18, "Yash Bose", "Kochi", 11, "2024-12-25", "repeat"],
    [52, "Harish Reddy", "unknown", 11, "2024-12-11", "repeat"],
    [47, "Janaki Patel", "Jaipur", 10, "2024-12-31", "repeat"],
    [2, "Manoj Menon", "Mumbai", 9, "2024-11-19", "repeat"],
    [15, "Kabir Gupta", "Chennai", 9, "2024-12-10", "repeat"],
    [17, "Charu Menon", "Kolkata", 9, "2024-12-23", "repeat"],
    [19, "Varun Pillai", "unknown", 9, "2024-12-03", "repeat"],
    [43, "Nisha Nair", "Hyderabad", 9, "2024-12-04", "repeat"],
    [13, "Yamini Reddy", "Chennai", 8, "2024-12-31", "repeat"],
    [25, "Farah Chopra", "Chennai", 8, "2024-12-15", "repeat"],
]
print("OK")

OK


## Self-Review Checklist

Check whether you can do each of these **without looking at the guide**. Anything you cannot, go back and
re-read that section — this list is the level in one page.

- [ ] Name the columns you want, rename one with `AS`, and say why `SELECT *` is a bad habit in saved code.
- [ ] Filter with `=`, `<>`, `<`, `>=`, and explain why text needs single quotes and numbers do not.
- [ ] Combine conditions with `AND` and `OR` and say why the brackets matter.
- [ ] Use `BETWEEN`, `IN` and `LIKE`, and say what `%` and `_` match.
- [ ] Explain why `WHERE city = NULL` returns nothing, and what to write instead.
- [ ] Explain why `WHERE city <> 'Chennai'` loses rows, and how to get them back.
- [ ] Sort by two columns in different directions, and say why a tiebreaker matters with `LIMIT`.
- [ ] Page through results with `LIMIT` and `OFFSET`.
- [ ] Use `DISTINCT` and `COUNT(DISTINCT x)`, and say how each treats `NULL`.
- [ ] Compute a percentage without falling into integer division.
- [ ] Slice a string with `SUBSTR` and `INSTR`, and join strings with `||`.
- [ ] Bucket dates into months with `strftime`, and add days with `date(d, '+7 days')`.
- [ ] Say what `COUNT(*)` and `COUNT(column)` each count.
- [ ] Write a `GROUP BY` and state the rule about what may appear in the `SELECT`.
- [ ] Say which conditions belong in `WHERE` and which in `HAVING`, and why.
- [ ] Write `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` and explain what it counts.
- [ ] Use `COALESCE` for a default and `NULLIF` to guard a division.
- [ ] Write an `INNER JOIN` with table aliases, and explain what happens to unmatched rows.
- [ ] Write the `LEFT JOIN ... WHERE right.pk IS NULL` shape and say what question it answers.
- [ ] Recite the clause execution order and use it to explain why `WHERE COUNT(*) > 5` fails.
- [ ] Create a table with `NOT NULL`, `DEFAULT`, `CHECK` and `REFERENCES`, and insert, update and delete rows.
- [ ] Say when a job belongs in SQL and when it belongs in pandas.

## What Next

`sql-intermediate` picks up where the two mini projects left off. It starts with the thing that quietly broke
your revenue number the first time you joined three tables — row fan-out — and goes on to subqueries, CTEs and
window functions, which are the tools that separate someone who can read a table from someone who can answer a
question.